In [11]:
import pandas as pd

# ========= 1. 读入四个模型的提交文件 =========
# 请根据你实际保存的文件名修改下面这四行
bert_path     = "bert_submission_b.csv"            # BERT
deberta_path  = "deberta_submission_b.csv"         # DeBERTa
roberta_path  = "roberta_submission_b.csv"     # RoBERTa-wwm-ext
roformer_path = "roformer_submission_b.csv"        # RoFormer

bert_df     = pd.read_csv(bert_path)
deberta_df  = pd.read_csv(deberta_path)
roberta_df  = pd.read_csv(roberta_path)
roformer_df = pd.read_csv(roformer_path)

# 确保 id 一致、类型统一（一般不需要，但稳一点）
for df in [bert_df, deberta_df, roberta_df, roformer_df]:
    df["id"] = df["id"].astype(int)

# 给每个 prob 列起一个不同的名字，方便后面加权
bert_df     = bert_df.rename(columns={"prob": "prob_bert"})
deberta_df  = deberta_df.rename(columns={"prob": "prob_deberta"})
roberta_df  = roberta_df.rename(columns={"prob": "prob_roberta"})
roformer_df = roformer_df.rename(columns={"prob": "prob_roformer"})

# ========= 2. 按 id 合并四个结果 =========
# inner join，保证只保留大家都有预测的样本
ensemble_df = (
    bert_df
    .merge(deberta_df, on="id")
    .merge(roberta_df, on="id")
    .merge(roformer_df, on="id")
)

print("合并后样本数：", len(ensemble_df))
ensemble_df.head()


合并后样本数： 1225


,id,prob_bert,prob_deberta,prob_roberta,prob_roformer
0,1,0.999043,0.999755,0.998239,0.997834
1,2,0.998587,0.999690,0.994314,0.997172
2,3,0.943301,0.141212,0.663287,0.875734
3,4,0.998833,0.999747,0.998020,0.998154
4,5,0.998999,0.999761,0.997876,0.998218


In [12]:
# ========= 3. 简单平均集成 =========
ensemble_df["prob_mean"] = ensemble_df[
    ["prob_bert", "prob_deberta", "prob_roberta", "prob_roformer"]
].mean(axis=1)

# ========= 4. 加权平均集成 =========
# 根据你之前验证集上的表现来设权重：
#   - RoBERTa-wwm-ext 最强 → 权重大一点
#   - RoFormer 第二
#   - BERT / DeBERTa 稍弱
w_roberta  = 0.4
w_roformer = 0.3
w_bert     = 0.2
w_deberta  = 0.1

ensemble_df["prob_weighted"] = (
    w_roberta  * ensemble_df["prob_roberta"] +
    w_roformer * ensemble_df["prob_roformer"] +
    w_bert     * ensemble_df["prob_bert"] +
    w_deberta  * ensemble_df["prob_deberta"]
)


In [13]:
# ========= 5. 导出简单平均版本 =========
submission_mean = ensemble_df[["id", "prob_mean"]].rename(
    columns={"prob_mean": "prob"}
)
submission_mean.to_csv(
    "ensemble4_mean_submission_a.csv",
    index=False,
    encoding="utf-8"
)
print("已保存：ensemble4_mean_submission_a.csv")

# ========= 6. 导出加权平均版本 =========
submission_weighted = ensemble_df[["id", "prob_weighted"]].rename(
    columns={"prob_weighted": "prob"}
)
submission_weighted.to_csv(
    "ensemble4_weighted_submission_a.csv",
    index=False,
    encoding="utf-8"
)
print("已保存：ensemble4_weighted_submission_a.csv")


已保存：ensemble4_mean_submission_a.csv
已保存：ensemble4_weighted_submission_a.csv


In [14]:
import pandas as pd
import numpy as np

# =========================
# 1) 填写你的 4 个 B 榜预测文件
#    每个文件都应包含列：id, prob
# =========================
files = {
    "bert":    "bert_submission_b.csv",
    "deberta": "deberta_submission_b.csv",
    "roberta": "roberta_submission_b.csv",
    "roformer":"roformer_submission_b.csv",
}

# =========================
# 2) 填写你这次重新训练得到的验证集 AUC（非常关键）
#    如果你暂时没有，就先用 None，后面会自动等权
# =========================
auc = {
    "bert":    0.9871,   # 改成你自己的
    "deberta": 0.9879,   # 改成你自己的
    "roberta": 0.9828,   # 改成你自己的
    "roformer":0.9900,   # 改成你自己的
}

# =========================
# 3) 权重策略：AUC 温度化
# =========================
ALPHA = 2.0   # 1.0/2.0/3.0 都可以试；时间紧建议 2.0
EPS = 1e-6

def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def get_weights(auc_dict, alpha=2.0):
    # 若 AUC 没填齐，则等权
    if any(v is None for v in auc_dict.values()):
        n = len(auc_dict)
        return {k: 1.0/n for k in auc_dict.keys()}

    s = {}
    for k, v in auc_dict.items():
        # AUC - 0.5 代表“优于随机的幅度”，避免把 0.5 附近的模型抬得太高
        s[k] = max(v - 0.5, 1e-6) ** alpha
    total = sum(s.values())
    return {k: s[k] / total for k in s.keys()}

weights = get_weights(auc, alpha=ALPHA)
print("Weights:", weights)

# =========================
# 4) 合并 4 个文件（按 id 对齐）
# =========================
dfs = []
for name, path in files.items():
    df = pd.read_csv(path).rename(columns={"prob": f"prob_{name}"})
    dfs.append(df)

merged = dfs[0]
for df in dfs[1:]:
    merged = merged.merge(df, on="id", how="inner")

# =========================
# 5) logit-space 加权融合
# =========================
Z = np.zeros(len(merged), dtype=float)
for name in files.keys():
    Z += weights[name] * logit(merged[f"prob_{name}"].values)

merged["prob"] = sigmoid(Z)

# =========================
# 6) 保存最终 B 榜提交文件
# =========================
out_path = "ensemble4_weighted_logit_submission_b.csv"
merged[["id", "prob"]].to_csv(out_path, index=False, encoding="utf-8")
print("Saved:", out_path)

# 可选：快速检查
print(merged[["id","prob"]].head())
print("prob range:", merged["prob"].min(), merged["prob"].max())


Weights: {'bert': 0.25014680414198853, 'deberta': 0.25096914771447637, 'roberta': 0.24574982794569322, 'roformer': 0.25313422019784193}
Saved: ensemble4_weighted_logit_submission_b.csv
   id      prob
0   1  0.999029
1   2  0.998377
2   3  0.713235
3   4  0.998983
4   5  0.999027
prob range: 0.0007474423413621387 0.9991355168032434


In [15]:
import pandas as pd

sub = pd.read_csv("ensemble4_weighted_logit_submission_b.csv")
print(sub.columns)
print(sub.shape)
print(sub["id"].is_unique, sub["id"].min(), sub["id"].max(), sub["id"].is_monotonic_increasing)
print(sub["prob"].isna().any(), sub["prob"].min(), sub["prob"].max())


Index(['id', 'prob'], dtype='object')
(1225, 2)
True 1 1225 True
False 0.0007474423413621 0.9991355168032434


In [16]:
import pandas as pd
import numpy as np

files = {
    "bert": "bert_submission_b.csv",
    "deberta": "deberta_submission_b.csv",
    "roberta": "roberta_submission_b.csv",
    "roformer": "roformer_submission_b.csv",
}

# 读入并按 id 合并
dfs = []
for name, path in files.items():
    df = pd.read_csv(path).rename(columns={"prob": f"prob_{name}"})
    dfs.append(df)

merged = dfs[0]
for df in dfs[1:]:
    merged = merged.merge(df, on="id", how="inner")

N = len(merged)

# 每个模型把 prob 转成 rank（1..N），再平均 rank
rank_sum = np.zeros(N, dtype=float)
for name in files.keys():
    rank_sum += merged[f"prob_{name}"].rank(method="average").values

rank_avg = rank_sum / len(files)

# 把 rank 映射回 (0,1)，单调映射不改变排序但更适合提交
merged["prob"] = (rank_avg - 0.5) / N

out_path = "ensemble4_rank_submission_b.csv"
merged[["id","prob"]].to_csv(out_path, index=False, encoding="utf-8")
print("Saved:", out_path, " range:", merged["prob"].min(), merged["prob"].max())


Saved: ensemble4_rank_submission_b.csv  range: 0.005510204081632653 0.9939795918367347
